# Notebook 05: anomalies in time, one-liners against a temporal autoencoder

**Class 03_02.** A value can be normal in itself and anomalous *at that moment*: 80 requests per second at noon is
normal, at 3 a.m. it is not (a **contextual** anomaly). Temporal detectors look at a **window** of the series.

**Data:** four artificial series of the Numenta Anomaly Benchmark (NAB, Lavin and Ahmad 2015), 5-minute samples over
14 days, a daily pattern (high by day, low at night), and one anomaly each:
`jumpsup` (a day goes too high), `jumpsdown` (too low), `nojump` (the daily rise does not happen), `flatmiddle` (the
signal stays flat). NAB labels a tolerance **window** around each anomaly.

**Protocol (as NAB):** the first 8 days are clean training data, day 9 sets the threshold (its maximum score + 10%),
days 1-14 are scored. We report whether the anomaly is **found** (an alarm inside the window), the **delay** from
the window start, and the **false alarms** (alarms outside the window).

In [ ]:
import os
import sys

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import pandas as pd
import polars as pl
from matplotlib import pyplot as plt

sys.path.insert(0, ".")
import plots_deep as pdp

import anomlib as an
import deeplib as dl

plt.rcParams["figure.dpi"] = 100
an.table_config()
D, W = dl.DAY, dl.DAY  # one day = 288 samples = the window length
SERIES = ["jumpsup", "jumpsdown", "nojump", "flatmiddle"]
fig, ax = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
for a, name in zip(ax, SERIES):
    _, te = dl.load_nab(name)
    a.plot(te.timestamp, te.value, lw=0.7)
    a.fill_between(te.timestamp, te.value.min(), te.value.max(), where=te.y == 1, color="tab:red", alpha=0.15)
    a.set_ylabel(name)
plt.tight_layout()
plt.show()
pdp.nab_series({n: dl.load_nab(n)[1] for n in SERIES})  # slide plots: assets/figures/plots/03/

## A. Three one-liners

* $|x_t|$ (z-score of the value): catches values outside the normal range only.
* **seasonal difference** $|x_t-x_{t-288}|$: compares with the same time yesterday.
* **moving standard deviation** over 1 hour: catches changes of variability.

## B. A convolutional autoencoder over one-day windows

Input: a window of 288 samples. Encoder: two 1-D convolutions with stride 2 (288 $\to$ 144 $\to$ 72 steps),
decoder: two transposed convolutions back to 288. The score of time step $t$ is the squared reconstruction error
**at $t$**, averaged over the 288 windows that contain it (averaging the *window* error instead smears an anomaly
over two days and causes hundreds of false alarms).

In [ ]:
ONE = {"|x|": lambda x: np.abs(x),
       "seasonal diff": lambda x: np.r_[np.full(D, np.nan), np.abs(x[D:] - x[:-D])],
       "moving std 1h": lambda x: pd.Series(x).rolling(12, min_periods=2).std().to_numpy()}  # fmt: skip
rows, curves = [], {}
for name in SERIES:
    _, te = dl.load_nab(name)
    v, y = te.value.to_numpy(), te.y.to_numpy()
    mu, sd = v[: 8 * D].mean(), v[: 8 * D].std()
    x = (v - mu) / sd
    m = dl.fit_window_ae(dl.conv_ae(W, 1), dl.windows(x[: 8 * D], W), epochs=30, seed=0)
    fs = dict(ONE)
    fs["conv AE"] = lambda x, m=m: dl.window_to_point(dl.window_error(m, dl.windows(x, W)), W, len(x))
    for k, f in fs.items():
        s = f(x)
        t = np.nanmax(s[8 * D : 9 * D]) * 1.1
        rows.append({"series": name, "detector": k, **dl.series_report(np.nan_to_num(s), y, t)})
        curves[(name, k)] = (s, t)
res = pl.DataFrame(rows)
print(res.pivot(on="detector", index="series", values="found"))
print(res.pivot(on="detector", index="series", values="false_alarms"))
print(res.pivot(on="detector", index="series", values="delay"))
pdp.nab_found(res.to_dicts())

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for a, k in zip(ax, ["seasonal diff", "conv AE"]):
    s, t = curves[("nojump", k)]
    a.plot(s, lw=0.7)
    a.axhline(t, color="k", ls="--")
    a.axvspan(*np.flatnonzero(dl.load_nab("nojump")[1].y)[[0, -1]], color="tab:red", alpha=0.15)
    a.set_ylabel(k)
ax[1].set_xlabel("time step (5 min)")
plt.tight_layout()
plt.show()
(s_d, t_d), (s_a, t_a) = curves[("nojump", "seasonal diff")], curves[("nojump", "conv AE")]
pdp.nab_scores(s_d, s_a, t_d, t_a, dl.load_nab("nojump")[1].y.to_numpy())

**Reading.**
* $|x|$ and the moving standard deviation find only the jump up: the other anomalies stay inside the normal range.
* The **seasonal difference finds all four**, with one weakness: the day *after* the anomaly is compared with the
  anomalous day, so it raises an "echo" of about 110 false alarms.
* The **conv AE finds all four with 1 to 35 false alarms**: it learned the shape of a normal day, so a missing rise
  or a flat stretch cannot be reconstructed.
* A one-liner that encodes the right prior (daily seasonality) gets most of the way. Benchmarks of time-series
  anomaly detection find the same: simple methods are often as good as deep ones (Wu and Keogh 2021; TSB-AD,
  Liu and Paparrizos 2024), and time-series *foundation models* used zero-shot do not beat moving-window one-liners
  (ICLR 2026). Always run the one-liners first.

## C. A recurrent autoencoder (LSTM), for comparison

In [ ]:
_, te = dl.load_nab("nojump")
v, y = te.value.to_numpy(), te.y.to_numpy()
x = (v - v[: 8 * D].mean()) / v[: 8 * D].std()
W2 = 96  # 8 hours: an LSTM over 288 steps is slow on a CPU
lstm = dl.fit_window_ae(dl.lstm_ae(W2, 1, units=32), dl.windows(x[: 8 * D], W2)[::4], epochs=20, seed=0)
s = dl.window_to_point(dl.window_error(lstm, dl.windows(x, W2)), W2, len(x))
print("LSTM AE on nojump:", dl.series_report(s, y, s[8 * D : 9 * D].max() * 1.1))

The LSTM AE (8-hour windows, to keep it fast on a CPU) does **not** find the missing rise: an 8-hour window does not
hold a whole day, so "no rise today" looks like a quiet morning. The window must cover the context that defines
normal; the architecture matters less.